In [ ]:
import json
from ollama import Client

---

In [ ]:
# === Настройка клиента Ollama ===
client = Client(host='http://localhost:11434')  # default Ollama port
model_name = "llama3.1:8b"  # или имя твоей модели

In [ ]:
# === Загрузи датасет ===
with open("../data/processed/conversations_2025-05-03.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
 # Формируем текст диалога
# prompt = ""
# for msg in conv:
#     role = msg["from"]
#     content = msg["value"]
#     prompt += f"{role}: {content}\n"
# prompt += "Леша:"  # Пусть модель продолжит за "Лешу"

conversation = data[len(data) // 2]
# prompt = f"Instruction: Твоя задача сделать диалог в формате вопрос/ответ. Иди по диалогу, бери вопрос Сони и на основе ответов Леши пиши ответ. Если ответ или вопрос разбиты на несколько сообщений, то объедени их в одно. Не придумывай вопросы и ответы! Используй только текст из диалога и стикеры! Твой ответ должен быть в формате json без лишних комментариев.\n\nContext: {conversation}. "

prompt = f"Instruction: Ты должен создать диалог в формате вопрос/ответ в виде массива JSON-объектов. Используй только текст из переданного диалога. \
1) Используй только реплики Сони в качестве вопросов. \
2) Используй только реплики Леши в качестве ответов. \
3) После вопроса Сони, должен следовать ответ Леши. \
3) Если вопрос или ответ разбиты на несколько сообщений подряд — объедини их в одно по смыслу. \
4) Не придумывай ничего от себя: только существующие сообщения и стикеры. \
6) Результат выдай строго в формате JSON: \
\
Context: {conversation}."
# === Отправляем в Ollama ===
response = client.generate(
    model=model_name,
    prompt=prompt,
)

In [ ]:
print(prompt)

In [ ]:
print(response.response)

In [ ]:
prompt

---

In [ ]:
with open("../data/processed/messages_2025-05-03.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
responses = []
for i, message in enumerate(data):
    if message['from'] == 'Соня' and '?' in message['text']:
        responses.append((message,data[i+1:i+10]))

In [ ]:
responses[0]

In [ ]:
data[0]['conversations'][0]

---

In [ ]:
with open("../data/processed/conversations_2025-05-03.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
delta_context = 10
delta_response = 10
max_responses = 2

new_data = []
for conv in data:
    for i, message in enumerate(conv['conversations']):
        if message['from'] == 'Соня' and '?' in message['value']:
            new_item = {}

            new_item["instruction"] = message['value']

            context = '\n'.join([f"{m['from']}:{m['value']}" for m in conv['conversations'][max(0, i - delta_context):i]])
            new_item["context"] = context

            response = []
            counter = 0
            for j in range(i+1, min(len(conv['conversations']), i+delta_response)):
                if conv['conversations'][j]['from'] == 'Леша':
                    response.append(conv['conversations'][j]['value'])
                    counter += 1
                if counter >= max_responses:
                    break
            new_item["response"] = '\n'.join(response)

            if response == []:
                continue

            new_data.append(new_item)

In [ ]:
new_data

In [ ]:
with open('../data/datasets/qa/qa_context.json', "w", encoding="utf-8") as f:
    json.dump(new_data, f, ensure_ascii=False, indent=2)

---

In [ ]:
with open("../data/processed/messages_2025-05-03.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
lesha = 0
sophi = 0
for message in data:
    if message['from'] == 'Леша':
        lesha += 1
    elif message['from'] == 'Соня':
        sophi += 1
    else:
        print('Unknowen user')

print(f'Леша: {lesha}')
print(f'Соня: {sophi}')

In [ ]:
ids = []
for i, message in enumerate(data):
    if message['from'] == 'Леша':
        ids.append(i)

In [ ]:
SIZE = 10_000
step = len(ids) // SIZE
window = 10

new_data = []
for i in range(0, len(ids), step):
    context = [data[id] for id in ids[max(0, i-window):i]]
    answer = data[ids[i]]
    new_data.append({
        "instruction": "\n".join([f"{message['from']}: {message['text']}" for message in context]),
        "input": "",
        "output": answer['text']
    })

In [ ]:
with open('../data/backup/dataset_plain.json', "w", encoding="utf-8") as f:
    json.dump(new_data, f, ensure_ascii=False, indent=2)